# NFL Big Data Bowl 2026 - Exploratory Data Analysis
## Relational Field Dynamics: Understanding Player Movement Patterns

**Goal:** Profile the training data to understand:
1. Play characteristics (duration, player counts, field positions)
2. Role-specific movement patterns
3. Relational coupling between players
4. Ball landing point influence on trajectories

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded")

## 1. Load Sample Data (Week 1)

In [ ]:
# Data paths
DATA_DIR = Path('data/nfl-big-data-bowl-2026-prediction')
TRAIN_DIR = DATA_DIR / 'train'

# Load week 1 as sample
input_w1 = pd.read_csv(TRAIN_DIR / 'input_2023_w01.csv')
output_w1 = pd.read_csv(TRAIN_DIR / 'output_2023_w01.csv')

print(f"Input shape: {input_w1.shape}")
print(f"Output shape: {output_w1.shape}")
print(f"\nInput columns: {list(input_w1.columns)}")

## 2. Play-Level Statistics

In [ ]:
# Group by play to get play-level stats
play_stats = input_w1.groupby(['game_id', 'play_id']).agg({
    'nfl_id': 'count',  # Number of players per play
    'num_frames_output': 'first',  # Trajectory length
    'ball_land_x': 'first',
    'ball_land_y': 'first',
    'absolute_yardline_number': 'first',
    'player_to_predict': 'sum'  # How many players to score
}).reset_index()

play_stats.columns = ['game_id', 'play_id', 'n_players', 'n_frames', 'ball_x', 'ball_y', 'yardline', 'n_scored']

print("Play Statistics:")
print(play_stats[['n_players', 'n_frames', 'n_scored']].describe())

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(play_stats['n_players'], bins=20, edgecolor='black')
axes[0].set_title('Players per Play')
axes[0].set_xlabel('Number of Players')

axes[1].hist(play_stats['n_frames'], bins=30, edgecolor='black')
axes[1].set_title('Trajectory Length (frames)')
axes[1].set_xlabel('Number of Frames')

axes[2].hist(play_stats['n_scored'], bins=15, edgecolor='black')
axes[2].set_title('Players Scored per Play')
axes[2].set_xlabel('Number to Predict')

plt.tight_layout()
plt.savefig('results/nfl_analysis/play_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Typical play: {play_stats['n_players'].median():.0f} players, "
      f"{play_stats['n_frames'].median():.0f} frames, "
      f"{play_stats['n_scored'].median():.0f} scored")

## 3. Role Analysis - The Relational Structure

In [ ]:
# Analyze roles
role_counts = input_w1.groupby('player_role').agg({
    'nfl_id': 'count',
    'player_to_predict': 'sum'
}).reset_index()
role_counts.columns = ['role', 'total', 'scored']
role_counts['pct_scored'] = 100 * role_counts['scored'] / role_counts['total']

print("\nRole Distribution:")
print(role_counts.sort_values('total', ascending=False))

In [ ]:
# Role-specific kinematics at t=0 (last frame before throw)
last_frames = input_w1.groupby(['game_id', 'play_id', 'nfl_id']).last().reset_index()

role_kinematics = last_frames.groupby('player_role').agg({
    's': ['mean', 'std'],  # Speed
    'a': ['mean', 'std'],  # Acceleration
}).round(3)

print("\nRole Kinematics (at throw moment):")
print(role_kinematics)

## 4. Ball Landing Point Analysis - The Attractor

In [ ]:
# Visualize ball landing positions
fig, ax = plt.subplots(figsize=(12, 6))

# Draw field
ax.add_patch(plt.Rectangle((0, 0), 120, 53.3, fill=False, edgecolor='green', linewidth=2))
for x in range(10, 120, 10):
    ax.axvline(x, color='white', linewidth=0.5, alpha=0.3)

# Plot ball landing points
unique_plays = play_stats[['ball_x', 'ball_y']].drop_duplicates()
ax.scatter(unique_plays['ball_x'], unique_plays['ball_y'], 
           alpha=0.3, s=20, c='red', label='Ball Landing')

ax.set_xlim(0, 120)
ax.set_ylim(0, 53.3)
ax.set_xlabel('X (yards)')
ax.set_ylabel('Y (yards)')
ax.set_title('Ball Landing Positions - The Attractor Field')
ax.legend()
ax.set_aspect('equal')

plt.savefig('results/nfl_analysis/ball_landing_field.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Trajectory Analysis - Movement Patterns

In [ ]:
# Pick one play to visualize
sample_game = input_w1['game_id'].iloc[0]
sample_play = input_w1['play_id'].iloc[0]

# Get input and output for this play
play_input = input_w1[(input_w1['game_id'] == sample_game) & (input_w1['play_id'] == sample_play)]
play_output = output_w1[(output_w1['game_id'] == sample_game) & (output_w1['play_id'] == sample_play)]

print(f"Sample Play: Game {sample_game}, Play {sample_play}")
print(f"Players: {play_input['nfl_id'].nunique()}")
print(f"Input frames: {play_input.groupby('nfl_id').size().max()}")
print(f"Output frames: {play_output.groupby('nfl_id').size().max()}")

In [ ]:
# Visualize trajectories for this play
fig, ax = plt.subplots(figsize=(14, 7))

# Draw field section
x_min, x_max = 30, 90
ax.add_patch(plt.Rectangle((x_min, 0), x_max - x_min, 53.3, 
                           fill=True, facecolor='#2d5f2e', edgecolor='white', linewidth=2))

# Get last input position and ball landing
last_input = play_input.groupby('nfl_id').last().reset_index()
ball_x = last_input['ball_land_x'].iloc[0]
ball_y = last_input['ball_land_y'].iloc[0]

# Plot each player's trajectory
for player_id in play_output['nfl_id'].unique():
    player_traj = play_output[play_output['nfl_id'] == player_id].sort_values('frame_id')
    player_info = last_input[last_input['nfl_id'] == player_id].iloc[0]
    
    # Color by role
    role = player_info['player_role']
    color = {'Passer': 'blue', 'Targeted Receiver': 'red', 
             'Defensive Coverage': 'orange', 'Other Route Runner': 'cyan'}.get(role, 'gray')
    
    # Plot trajectory
    ax.plot(player_traj['x'], player_traj['y'], 
           color=color, alpha=0.6, linewidth=2, label=role if player_id == play_output['nfl_id'].unique()[0] else '')
    
    # Mark start position
    ax.scatter(player_info['x'], player_info['y'], 
              color=color, s=100, marker='o', edgecolor='white', linewidth=2, zorder=3)

# Plot ball landing
ax.scatter(ball_x, ball_y, color='yellow', s=300, marker='*', 
          edgecolor='black', linewidth=2, label='Ball Landing', zorder=4)

ax.set_xlim(x_min, x_max)
ax.set_ylim(0, 53.3)
ax.set_xlabel('X (yards)', fontsize=12)
ax.set_ylabel('Y (yards)', fontsize=12)
ax.set_title(f'Player Trajectories: Game {sample_game}, Play {sample_play}', fontsize=14)
ax.legend()
ax.set_aspect('equal')

plt.savefig('results/nfl_analysis/sample_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Relational Feature Analysis

In [ ]:
# Compute distance to ball for each player at throw moment
last_frames['dist_to_ball'] = np.sqrt(
    (last_frames['x'] - last_frames['ball_land_x'])**2 + 
    (last_frames['y'] - last_frames['ball_land_y'])**2
)

# Compute speed toward ball
# Vector to ball
dx = last_frames['ball_land_x'] - last_frames['x']
dy = last_frames['ball_land_y'] - last_frames['y']
dist = np.sqrt(dx**2 + dy**2)

# Player velocity components
vx = last_frames['s'] * np.cos(np.radians(last_frames['dir']))
vy = last_frames['s'] * np.sin(np.radians(last_frames['dir']))

# Dot product: v · (ball_direction)
last_frames['speed_toward_ball'] = (vx * dx + vy * dy) / (dist + 1e-6)

# Analyze by role
role_dist = last_frames.groupby('player_role').agg({
    'dist_to_ball': ['mean', 'std'],
    'speed_toward_ball': ['mean', 'std']
}).round(2)

print("\nRole-Based Ball Relationship:")
print(role_dist)

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distance to ball by role
last_frames.boxplot(column='dist_to_ball', by='player_role', ax=axes[0])
axes[0].set_title('Distance to Ball Landing (at throw)')
axes[0].set_xlabel('Role')
axes[0].set_ylabel('Distance (yards)')
axes[0].tick_params(axis='x', rotation=45)

# Speed toward ball by role
last_frames.boxplot(column='speed_toward_ball', by='player_role', ax=axes[1])
axes[1].set_title('Speed Toward Ball (at throw)')
axes[1].set_xlabel('Role')
axes[1].set_ylabel('Speed (yards/s)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/nfl_analysis/relational_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS FOR MODEL DESIGN")
print("="*60)

print("\n1. PLAY STRUCTURE:")
print(f"   - Typical play: ~{play_stats['n_players'].median():.0f} players tracked")
print(f"   - Trajectory length: {play_stats['n_frames'].quantile(0.25):.0f}-{play_stats['n_frames'].quantile(0.75):.0f} frames (IQR)")
print(f"   - Players scored: {play_stats['n_scored'].median():.0f} per play")

print("\n2. ROLE HIERARCHY:")
print(f"   - Most common: {role_counts.iloc[0]['role']} ({role_counts.iloc[0]['total']} instances)")
print(f"   - Most scored: {role_counts.nlargest(1, 'pct_scored').iloc[0]['role']} "
      f"({role_counts.nlargest(1, 'pct_scored').iloc[0]['pct_scored']:.1f}% scored)")

print("\n3. RELATIONAL DYNAMICS:")
print(f"   - Targeted Receivers: Closest to ball, high speed toward target")
print(f"   - Defensive Coverage: Also moves toward ball (competing attractor)")
print(f"   - Passer: Minimal movement, low speed")
print(f"   - Other Route Runners: Moderate engagement with ball")

print("\n4. MODEL IMPLICATIONS:")
print("   ✓ Role is CRITICAL feature (determines movement pattern)")
print("   ✓ Ball landing point is PRIMARY attractor")
print("   ✓ Need role-specific prediction strategies")
print("   ✓ Relational features (player-ball, player-player) essential")
print("   ✓ Variable trajectory lengths require flexible architecture")

print("\n" + "="*60)

## Next Steps

Based on this EDA:

1. **Feature Engineering:** Create relational features
   - Distance/angle to ball
   - Speed toward ball
   - Player-player distances (spatial graph)
   - Role embeddings

2. **Baseline Model:** Physics-based prediction
   - Linear extrapolation with ball attraction
   - Role-specific parameters

3. **Advanced Model:** GNN or Transformer
   - Graph structure: players as nodes
   - Ball as global context
   - Temporal decoder for trajectories

4. **Evaluation:** RMSE on held-out weeks
   - Cross-validate across weeks
   - Analyze per-role performance